# Optimize 研究

通过插件将上游项目安装为 dev 依赖。表单构造本项目的 `OptimizeAnalysisParams`，继承当前算法的 `OptimizeParams`。只在此 Notebook 合并上游参数，算法包不依赖上游项目。Scheme 各环节读取统一的同名字段；同名取值不一致时明确报错，不静默覆盖。

首个代码单元格优先加载当前项目 `src` 源码；修改源码后重跑此格，不使用环境中旧安装包。请在当前项目目录运行内核。

In [ ]:
from pathlib import Path
import importlib
import sys
import tomllib

project_root = Path.cwd().resolve()
pyproject_path = project_root / "pyproject.toml"
if not pyproject_path.is_file():
    raise FileNotFoundError("请将内核工作目录设为包含 pyproject.toml 和 src 的当前项目目录。")
with pyproject_path.open("rb") as project_file:
    project_config = tomllib.load(project_file)
package_name = (
    project_config.get("tool", {}).get("uv", {}).get("build-backend", {}).get("module-name")
    or project_config.get("project", {}).get("name", "").replace("-", "_")
)
src_path = (project_root / "src").resolve()
if not package_name or not src_path.is_dir():
    raise FileNotFoundError("当前项目缺少包名配置或 src 目录；请检查 pyproject.toml 和工作目录。")
package_path = src_path.joinpath(*package_name.split("."))
if not package_path.is_dir() and not package_path.with_suffix(".py").is_file():
    raise FileNotFoundError(f"找不到当前项目源码：{package_path}；请检查包名配置。")
sys.path[:] = [str(src_path)] + [path for path in sys.path if path != str(src_path)]
importlib.invalidate_caches()
for loaded_name in list(sys.modules):
    if loaded_name == package_name or loaded_name.startswith(package_name + "."):
        del sys.modules[loaded_name]

from importlib import import_module
from optimize import OptimizeAlgo, OptimizeAnalysisParams, OptimizeReportForm
from scheme import StockPool, algo_options
from scheme.execute.strategy.assembly import parameter_type, project_parameters

model_options = algo_options("model")

model_options

## 选择与配置上游
先查看上一格的可用入口，然后显式填写下方的 `*_entry`，即使只有一个选项也不会自动选中。没有选项时请先安装兼容上游；多个选项时由研究者决定。`*_overrides` 只填写算法特有字段（如 Model 的 `n_select`），不要覆盖公共研究设置。`start`、`end`、`universe` 等从当前分析参数传播给上游；参数类型从选中的 Algo 泛型解析，无需假定包内导出的 Params 名称。使用 Scheme 1.2.1+ 的参数投影辅助函数。

In [ ]:
def selected_upstream(kind, entry, options):
    if not options:
        raise ValueError(
            f"没有可用的 {kind} 上游项目。请通过插件安装兼容的 dev 依赖，"
            "重启内核后重新运行选项单元格。"
        )
    if entry not in options:
        raise ValueError(f"请显式设置 {kind}_entry；可用入口：{list(options)}")
    module, name = entry.split(":")
    return parameter_type(getattr(import_module(module), name))


def merge_upstream(analysis, upstream):
    # 仅覆盖算法特有字段。
    common = {
        "start", "end", "universe", "market_data", "symbols", "benchmark",
        "batch_days", "config", "model", "optimize", "control", "execution",
    }
    values = dict(analysis)  # 保留嵌套 Universe。
    for parameter_model, overrides in upstream:
        forbidden = common & overrides.keys()
        unknown = overrides.keys() - parameter_model.model_fields.keys()
        if forbidden or unknown:
            raise ValueError(f"上游只能覆盖已声明的算法字段：{sorted(forbidden | unknown)}")
        # 传播公共研究设置，包括必填日期与股票池。
        source = dict(analysis)
        for name in overrides:
            if name not in type(analysis).model_fields:
                source.pop(name, None)
        upstream_params = project_parameters(parameter_model, source, overrides=overrides)
        for name, value in dict(upstream_params).items():
            if name in values and values[name] != value:
                raise ValueError(f"上游同名参数冲突：{name}；请统一各环节的取值。")
            values[name] = value
    return type(analysis).model_validate(values)


In [ ]:
model_entry = None  # 显式填入 model_options 中的入口字符串
model_overrides = {}  # 例如 {"n_select": 10}，仅算法特有字段

ModelParams = selected_upstream("model", model_entry, model_options)

form = OptimizeReportForm(
    start="2026-06-01",
    end="2026-06-06",
    pool=StockPool.SSE50,
    model=model_entry,
)


In [ ]:
params: OptimizeAnalysisParams = form.build()
params = merge_upstream(params, [
    (ModelParams, model_overrides),
])
params

In [ ]:
report = params.run(OptimizeAlgo)
report.show()